# Simulating the Amplitude-Based Collisionless QLBM

In [ ]:
from qiskit_aer import AerSimulator

from qlbm.components import (
    CQLBM,
    ABGridMeasurement,
    ABParallelDiscreteUniformInitialConditions,
    EmptyPrimitive,
)
from qlbm.infra import QiskitRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 16, "y": 16}, "velocities": "d2q9"},
        "geometry": [
            {"shape": "cuboid", "x": [10, 13], "y": [6, 14], "boundary": "bounceback"},
        ],
    }
)

output_dir = "qlbm-output/ab-pic-d2q9-16x16-1-obstacle-qiskit"
create_directory_and_parents(output_dir)

In [ ]:
# Set the number of marker registers, such that we can have up to 2 parallel initial conditions
lattice.set_num_marker_qubits(1)

In [ ]:
# ICs moving E and NE, assembled on a rectangle [0, 1, 2, 3] x [0, ..., 15] on marker |0>
# and W and SW, assembled on a rectangle [0, ..., 15] x [0, 1, 2, 3] on marker |1>
ics = ABParallelDiscreteUniformInitialConditions(
    lattice,
    [[1, 5], [3, 7]],
    [([0, 1], [0, 1, 2, 3]), ([0, 1, 2, 3], [0, 1])],
)

ics.draw("mpl")

In [ ]:
cfg = SimulationConfig(
    initial_conditions=ics,
    algorithm=CQLBM(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    target_platform="QISKIT",
    compiler_platform="QISKIT",
    optimization_level=0,
    statevector_sampling=True,
    execution_backend=AerSimulator(method="statevector"),
    sampling_backend=AerSimulator(method="statevector"),
)

In [ ]:
cfg.prepare_for_simulation()

In [ ]:
# Number of shots to simulate for each timestep when running the circuit
NUM_SHOTS = 2**12

# Number of timesteps to simulate
NUM_STEPS = 20

In [ ]:
runner = QiskitRunner(
    cfg,
    lattice,
)


# Simulate the circuits using both snapshots
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)